# TRIAGE-EG Notebook 37 — Dual-T4 ASR v1.2

ASR-only fail-closed Save Version. It inventories all audio, benchmarks batch 4/8/16 by end-to-end RTF, benchmarks the selected configuration on one versus two GPUs, and launches the full corpus only when projected ETA is at most 10 hours. It does not run XCLIP, DINO, OCR, Qwen, Event Graph, or GT.

In [ ]:
import os
from pathlib import Path
REPO_URL="https://github.com/Irthn1311/AIC2026_TeamPTK_SGU.git"
REPO_REF="TRIAGEEG"
ANCHOR="d338d8d809bbb9e057e18becc607af2eb3bea254"
REPO_DIR=Path(os.environ.get("AIC_REPO_DIR","/kaggle/working/AIC2026_TeamPTK_SGU"))
RAW_INPUT=Path(os.environ.get("AIC_DATA_ROOT","/kaggle/input/datasets/nadkli/dataset-aic"))
WHISPER_INPUT=Path(os.environ.get("AIC_WHISPER_ROOT","/kaggle/input/datasets/irthn1311/fs1-whisper-large-v3-turbo-asset"))
RESUME_INPUT=Path(os.environ["AIC_ASR_RESUME_ROOT"]) if os.environ.get("AIC_ASR_RESUME_ROOT") else None
OUTPUT_ROOT=Path("/kaggle/working/triage_eg_asr_dual_t4_v12")
OUTPUT_ROOT.mkdir(parents=True,exist_ok=True)
OUTPUT_ZIP=Path("/kaggle/working/asr_dual_t4_bundle_v12.zip")
BENCHMARK_ZIP=Path("/kaggle/working/asr_dual_t4_benchmark_v12.zip")
EXPECTED_VIDEO_COUNT=873
BENCHMARK_VIDEO_COUNT=20
MAX_PROJECTED_HOURS=10.0
print({"required_inputs":{"raw_dataset":str(RAW_INPUT),"whisper_turbo_offline_asset":str(WHISPER_INPUT),"optional_resume_bundle":str(RESUME_INPUT) if RESUME_INPUT else None},"internet_required":"ONLY_FOR_GIT_CLONE_OR_EXPLICIT_REFRESH","gpu_policy":"AUTO_1_OR_2_T4; ONE_PROCESS_AND_MODEL_REPLICA_PER_GPU","scope":"ASR_ONLY_NO_XCLIP_DINO_OCR_QWEN_GRAPH_GT","benchmark_zip":str(BENCHMARK_ZIP),"full_output_zip":str(OUTPUT_ZIP)})


In [ ]:
import json,subprocess,sys,torch
if not (REPO_DIR/".git").is_dir(): subprocess.run(["git","clone","--branch",REPO_REF,"--single-branch",REPO_URL,str(REPO_DIR)],check=True)
subprocess.run(["git","fetch","origin",REPO_REF],cwd=REPO_DIR,check=True)
subprocess.run(["git","checkout","--detach","FETCH_HEAD"],cwd=REPO_DIR,check=True)
HEAD=subprocess.check_output(["git","rev-parse","HEAD"],cwd=REPO_DIR,text=True).strip()
if subprocess.run(["git","merge-base","--is-ancestor",ANCHOR,HEAD],cwd=REPO_DIR).returncode: raise RuntimeError(f"ASR_V12_LINEAGE_FAIL anchor={ANCHOR} HEAD={HEAD}")
sys.path.insert(0,str(REPO_DIR/"src"))
VISIBLE_GPU_COUNT=torch.cuda.device_count()
if VISIBLE_GPU_COUNT<1: raise RuntimeError("ASR_V12_REQUIRES_AT_LEAST_ONE_CUDA_GPU")
GPU_COUNT=min(VISIBLE_GPU_COUNT,2)
RUNTIME={"torch":torch.__version__,"cuda":torch.version.cuda,"visible_gpu_count":VISIBLE_GPU_COUNT,"used_gpu_count":GPU_COUNT,"devices":[{"index":index,"name":torch.cuda.get_device_name(index),"total_vram_bytes":torch.cuda.get_device_properties(index).total_memory} for index in range(VISIBLE_GPU_COUNT)],"cpu_count":os.cpu_count(),"decode_workers_total":min(4,max(1,(os.cpu_count() or 2)//2))}
print({"HEAD":HEAD,"anchor_is_ancestor":True,"runtime":RUNTIME})


In [ ]:
def bounded(root,name,max_depth=6):
    output=[]; root=Path(root)
    if not root.exists(): return output
    for directory,subdirs,files in os.walk(root):
        current=Path(directory); depth=len(current.relative_to(root).parts); subdirs[:]=[] if depth>=max_depth else [item for item in subdirs if item not in {".cache","blobs","snapshots"}]
        if name in files: output.append(current/name)
    return output
def unique(values,label):
    values=sorted(set(Path(value).resolve() for value in values))
    if len(values)!=1: raise RuntimeError(f"Expected one {label}; found {values}")
    return values[0]
WHISPER_ROOT=unique(bounded(WHISPER_INPUT,"config.json") or bounded("/kaggle/input","config.json"),"Whisper config.json").parent
from triage_eg.data.stage0_audit.asset_resolver import discover_layout
video_parts,_=discover_layout(RAW_INPUT)
print({"raw_dataset":str(RAW_INPUT.resolve()),"whisper_asset":str(WHISPER_ROOT),"discovered_videos":len(video_parts)})


In [ ]:
test_env=os.environ.copy(); test_env["PYTHONPATH"]=str(REPO_DIR/"src")+(os.pathsep+test_env["PYTHONPATH"] if test_env.get("PYTHONPATH") else "")
test=subprocess.run([sys.executable,"-m","pytest","tests/unit/fs1_v11/test_asr_v12.py","tests/unit/fs1_v11/test_completion_core.py","-q"],cwd=REPO_DIR,env=test_env,capture_output=True,text=True)
TEST_SUMMARY={"returncode":test.returncode,"stdout_tail":test.stdout.splitlines()[-20:],"stderr_tail":test.stderr.splitlines()[-20:]}
if test.returncode: raise RuntimeError(TEST_SUMMARY)
print(TEST_SUMMARY)


In [ ]:
# Full-corpus ffprobe inventory and deterministic LPT duration balancing.
from concurrent.futures import ThreadPoolExecutor
from triage_eg.fs1_v11.asr_v12 import atomic_write_json,atomic_write_jsonl,lpt_partition,probe_audio,representative_sample
if len(video_parts)!=EXPECTED_VIDEO_COUNT: raise RuntimeError(f"ASR_INVENTORY_VIDEO_COUNT_FAIL expected={EXPECTED_VIDEO_COUNT} actual={len(video_parts)}")
with ThreadPoolExecutor(max_workers=RUNTIME["decode_workers_total"]) as pool:
    INVENTORY=list(pool.map(lambda item:probe_audio(item[0],RAW_INPUT/item[1]/"video"/f"{item[0]}.mp4"),sorted(video_parts.items())))
atomic_write_jsonl(OUTPUT_ROOT/"asr_audio_inventory_v12.jsonl",INVENTORY)
probe_failures=[row for row in INVENTORY if row["probe_status"]=="FAILED"]
audio_rows=[row for row in INVENTORY if row["has_audio"] and row["probe_status"]=="PASS"]
if probe_failures: raise RuntimeError({"ASR_FFPROBE_FAILURES":probe_failures})
if not audio_rows: raise RuntimeError("ASR_NO_AUDIO_BEARING_VIDEOS")
FULL_SHARDS=lpt_partition(INVENTORY,GPU_COUNT)
FULL_MANIFESTS=[]
for shard in FULL_SHARDS:
    path=OUTPUT_ROOT/f"asr_shard_{shard['shard_id']}_manifest.json"; atomic_write_json(path,shard); FULL_MANIFESTS.append(path)
TOTAL_AUDIO_SECONDS=sum(row["duration_seconds"] for row in audio_rows)
SHARD_BALANCE={"total_audio_hours":TOTAL_AUDIO_SECONDS/3600,"shards":[{"shard_id":row["shard_id"],"video_count":row["video_count"],"audio_hours":row["total_audio_seconds"]/3600,"imbalance_percent":row["imbalance_percent"]} for row in FULL_SHARDS]}
atomic_write_json(OUTPUT_ROOT/"asr_shard_balance_v12.json",SHARD_BALANCE)
print(SHARD_BALANCE)


In [ ]:
# Deterministic 20-video single-GPU benchmark for batch 4/8/16. Selection uses end-to-end RTF.
from triage_eg.fs1_v11.asr_v12 import launch_workers,read_jsonl,select_lowest_rtf
BENCHMARK_ROWS=representative_sample(INVENTORY,BENCHMARK_VIDEO_COUNT)
BENCHMARK_MANIFEST=OUTPUT_ROOT/"benchmark_20_manifest.json"
atomic_write_json(BENCHMARK_MANIFEST,{"shard_id":0,"video_count":len(BENCHMARK_ROWS),"total_audio_seconds":sum(row["duration_seconds"] for row in BENCHMARK_ROWS),"videos":BENCHMARK_ROWS})
SINGLE_REPORTS=[]; SINGLE_CHECKPOINTS={}
for batch_size in (4,8,16):
    checkpoint=OUTPUT_ROOT/f"benchmark_single_bs{batch_size}.jsonl"; progress=OUTPUT_ROOT/f"benchmark_single_bs{batch_size}_progress.json"
    for stale in (checkpoint,progress,progress.with_name(progress.stem+"_report.json")):
        if stale.exists(): stale.unlink()
    try: report=launch_workers(REPO_DIR,[BENCHMARK_MANIFEST],[checkpoint],[progress],WHISPER_ROOT,batch_size,[0],benchmark=True)
    except RuntimeError as error:
        report={"gpu_count":1,"batch_size":batch_size,"video_count":BENCHMARK_VIDEO_COUNT,"status":"INVALID_WORKER_ERROR","error":str(error),"rtf":float("inf")}
    SINGLE_REPORTS.append(report); SINGLE_CHECKPOINTS[batch_size]=checkpoint
SELECTED_SINGLE=select_lowest_rtf(SINGLE_REPORTS); SELECTED_BATCH=int(SELECTED_SINGLE["batch_size"])
REFERENCE={row["video_id"]:row for row in read_jsonl(SINGLE_CHECKPOINTS[4])}; SELECTED={row["video_id"]:row for row in read_jsonl(SINGLE_CHECKPOINTS[SELECTED_BATCH])}
from triage_eg.fs1_v11.asr_v12 import material_consistency
QUALITY=[{"video_id":video_id,"language_equal":REFERENCE[video_id].get("language")==SELECTED[video_id].get("language"),"token_jaccard":material_consistency(REFERENCE[video_id],SELECTED[video_id])} for video_id in sorted(REFERENCE) if REFERENCE[video_id].get("status")==SELECTED.get(video_id,{}).get("status")=="PASS"]
if len(QUALITY)!=BENCHMARK_VIDEO_COUNT or min(row["token_jaccard"] for row in QUALITY)<0.75: raise RuntimeError({"ASR_BATCH_QUALITY_CONSISTENCY_FAIL":QUALITY})
print({"single_gpu_batch_reports":SINGLE_REPORTS,"selected_batch":SELECTED_BATCH,"quality":QUALITY})


In [ ]:
# Same representative sample on all visible/used GPUs.
BENCH_SHARDS=lpt_partition(BENCHMARK_ROWS,GPU_COUNT); bench_manifests=[]; bench_checkpoints=[]; bench_progress=[]
for shard in BENCH_SHARDS:
    shard_id=shard["shard_id"]; manifest=OUTPUT_ROOT/f"benchmark_dual_shard_{shard_id}_manifest.json"; checkpoint=OUTPUT_ROOT/f"benchmark_dual_shard_{shard_id}.jsonl"; progress=OUTPUT_ROOT/f"benchmark_dual_shard_{shard_id}_progress.json"
    atomic_write_json(manifest,shard)
    for stale in (checkpoint,progress,progress.with_name(progress.stem+"_report.json")):
        if stale.exists(): stale.unlink()
    bench_manifests.append(manifest); bench_checkpoints.append(checkpoint); bench_progress.append(progress)
DUAL_REPORT=launch_workers(REPO_DIR,bench_manifests,bench_checkpoints,bench_progress,WHISPER_ROOT,SELECTED_BATCH,list(range(GPU_COUNT)),benchmark=True)
if DUAL_REPORT["transcript_success_count"]!=BENCHMARK_VIDEO_COUNT or DUAL_REPORT["timestamp_valid_count"]!=BENCHMARK_VIDEO_COUNT: raise RuntimeError({"ASR_DUAL_BENCHMARK_VALIDITY_FAIL":DUAL_REPORT})
PROJECTED_SECONDS=TOTAL_AUDIO_SECONDS*DUAL_REPORT["rtf"]; PROJECTED_HOURS=PROJECTED_SECONDS/3600
PERFORMANCE_REPORT={"runtime":RUNTIME,"corpus":SHARD_BALANCE,"benchmark_video_count":BENCHMARK_VIDEO_COUNT,"benchmark_audio_hours":sum(row["duration_seconds"] for row in BENCHMARK_ROWS)/3600,"single_gpu_batch_reports":SINGLE_REPORTS,"selected_batch_size":SELECTED_BATCH,"selected_single_gpu_rtf":SELECTED_SINGLE["rtf"],"selected_multi_gpu_rtf":DUAL_REPORT["rtf"],"multi_gpu_report":DUAL_REPORT,"quality_comparison":QUALITY,"projected_full_wall_seconds":PROJECTED_SECONDS,"projected_full_wall_hours":PROJECTED_HOURS,"eta_gate_hours":MAX_PROJECTED_HOURS,"full_run_authorized":PROJECTED_HOURS<=MAX_PROJECTED_HOURS}
atomic_write_json(OUTPUT_ROOT/"asr_performance_report_v12.json",PERFORMANCE_REPORT)
import shutil
shutil.make_archive(str(BENCHMARK_ZIP.with_suffix("")),"zip",OUTPUT_ROOT)
print({"ASR_20_VIDEO_BENCHMARK":PERFORMANCE_REPORT,"benchmark_download_zip":str(BENCHMARK_ZIP)})
RUN_FULL=bool(PERFORMANCE_REPORT["full_run_authorized"])
if not RUN_FULL: print({"status":"STOP_PROJECTED_ETA_GT_10H","projected_hours":PROJECTED_HOURS,"full_run_launched":False})


In [ ]:
# Full run is reached only through the measured <=10h ETA gate.
FULL_REPORT=None
if RUN_FULL:
    full_checkpoints=[OUTPUT_ROOT/f"asr_shard_{index}_v12.jsonl" for index in range(GPU_COUNT)]; full_progress=[OUTPUT_ROOT/f"asr_shard_{index}_progress_v12.json" for index in range(GPU_COUNT)]
    if RESUME_INPUT and RESUME_INPUT.exists():
        for index,target in enumerate(full_checkpoints):
            matches=sorted(RESUME_INPUT.rglob(f"asr_shard_{index}_v12.jsonl"))
            if len(matches)>1: raise RuntimeError(f"AMBIGUOUS_RESUME_SHARD_{index}: {matches}")
            if matches and not target.exists(): shutil.copy2(matches[0],target)
    FULL_REPORT=launch_workers(REPO_DIR,FULL_MANIFESTS,full_checkpoints,full_progress,WHISPER_ROOT,SELECTED_BATCH,list(range(GPU_COUNT)),benchmark=False)
    print({"ASR_FULL_RUN_FINISHED":FULL_REPORT})


In [ ]:
# Merge, hard gates, reload, and ASR-only bundle. No other modality and no GT.
from triage_eg.fs1_v11.asr_v12 import lexical_index,merge_shards,timestamps_monotonic
if RUN_FULL:
    MERGED,MERGE_DIAGNOSTICS=merge_shards(INVENTORY,full_checkpoints)
    PASS_ROWS=[row for row in MERGED if row.get("status")=="PASS"]; FAIL_ROWS=[row for row in MERGED if row.get("status")!="PASS"]; NONEMPTY=[row for row in PASS_ROWS if any(segment.get("normalized_text") for segment in row.get("segments",[]))]
    PASS_RATE=len(PASS_ROWS)/max(len(MERGED),1); LEXICAL_INDEX=lexical_index(MERGED)
    if PASS_RATE<0.95: raise RuntimeError({"ASR_PASS_RATE_GATE_FAIL":PASS_RATE,"failures":FAIL_ROWS[:50]})
    if len(NONEMPTY)<100 or not LEXICAL_INDEX: raise RuntimeError({"ASR_NONEMPTY_OR_LEXICAL_GATE_FAIL":{"nonempty":len(NONEMPTY),"terms":len(LEXICAL_INDEX)}})
    if not all(timestamps_monotonic(row["segments"]) for row in PASS_ROWS): raise RuntimeError("ASR_MERGED_TIMESTAMP_GATE_FAIL")
    atomic_write_jsonl(OUTPUT_ROOT/"asr_transcripts_v12.jsonl",MERGED); atomic_write_json(OUTPUT_ROOT/"asr_lexical_index_v12.json",LEXICAL_INDEX)
    RELOADED=read_jsonl(OUTPUT_ROOT/"asr_transcripts_v12.jsonl")
    if len(RELOADED)!=len(MERGED) or [row["video_id"] for row in RELOADED]!=[row["video_id"] for row in MERGED]: raise RuntimeError("ASR_MERGED_RELOAD_GATE_FAIL")
    PERFORMANCE_REPORT.update({"actual_full_run":FULL_REPORT,"actual_full_rtf":FULL_REPORT["rtf"],"merge":MERGE_DIAGNOSTICS,"pass_rate":PASS_RATE,"pass_count":len(PASS_ROWS),"failure_count":len(FAIL_ROWS),"nonempty_video_count":len(NONEMPTY),"lexical_term_count":len(LEXICAL_INDEX),"status":"PASS"})
    atomic_write_json(OUTPUT_ROOT/"asr_performance_report_v12.json",PERFORMANCE_REPORT); atomic_write_json(OUTPUT_ROOT/"tests_summary.json",TEST_SUMMARY)
    shutil.make_archive(str(OUTPUT_ZIP.with_suffix("")),"zip",OUTPUT_ROOT)
    print({"status":"PASS","download_zip":str(OUTPUT_ZIP),"performance":PERFORMANCE_REPORT})
else:
    print({"status":"STOPPED_AFTER_BENCHMARK","benchmark_zip":str(BENCHMARK_ZIP),"full_bundle_created":False})
